# ML-05 - Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NameRectified/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** - each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it - engineered features, categorical handling, fills.*

In [2]:
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

env_path = Path('../../.env')
if env_path.exists():
    for line in env_path.read_text().strip().split('\n'):
        if '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

features = con.sql(f"""
    SELECT f.content_hash_id,
           SUM(f.gsc_impressions) AS impressions_fw,
           SUM(f.gsc_clicks) AS clicks_fw,
           AVG(f.gsc_avg_position) AS avg_pos_fw,
           STDDEV_SAMP(f.gsc_avg_position) AS pos_volatility_fw,
           SUM(f.ga4_sessions) AS sessions_fw,
           SUM(f.ga4_engaged_sessions) AS engaged_sessions_fw,
           c.content_type,
           c.main_intent
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-03-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, c.content_type, c.main_intent
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

features['ctr_fw'] = features['clicks_fw'] / features['impressions_fw'] * 100
features['engagement_rate_fw'] = features['engaged_sessions_fw'] / features['sessions_fw'] * 100

def assign_tier(pos):
    if pos <= 3: return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

features['position_tier'] = features['avg_pos_fw'].apply(assign_tier)
tier_med = features.groupby('position_tier').apply(
    lambda g: g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100 if g['impressions_fw'].sum() > 0 else 0,
    include_groups=False
)
features['tier_median_ctr'] = features['position_tier'].map(tier_med)
features['tier_ctr_gap'] = features['tier_median_ctr'] - features['ctr_fw']

features['content_type'] = features['content_type'].fillna('unknown')
features['main_intent'] = features['main_intent'].fillna('unknown')
features = features.fillna(0)

print(f'Feature vector: {len(features):,} rows x {len(features.columns)} columns')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector: 90,235 rows x 14 columns


,content_hash_id,impressions_fw,clicks_fw,avg_pos_fw,pos_volatility_fw,sessions_fw,engaged_sessions_fw,content_type,main_intent,ctr_fw,engagement_rate_fw,position_tier,tier_median_ctr,tier_ctr_gap
0,content_0215a034c79d2bab,650.0,1.0,38.048692,23.866893,0.0,0.0,keyword article,informational,0.153846,0.0,page_3_5,0.162061,0.008215
1,content_07418784c2a5a884,940.0,1.0,15.267078,5.398516,0.0,0.0,keyword article,informational,0.106383,0.0,striking,0.300992,0.194609
2,content_082bf9b8307c6542,138.0,0.0,51.100425,25.333706,0.0,0.0,keyword article,informational,0.000000,0.0,deep,0.048607,0.048607
3,content_0cb1b73128cbe2e0,10176.0,17.0,8.126085,2.546171,0.0,0.0,keyword article,informational,0.167060,0.0,page_1,0.331373,0.164313
4,content_0ded34cdbb938acc,138.0,0.0,27.494966,16.691977,0.0,0.0,keyword article,informational,0.000000,0.0,page_3_5,0.162061,0.162061


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handled by | Available when? |
|---|---|---|---|
| impressions_fw | Total GSC impressions in the 60 days before March | fillna(0) | Already recorded by March 1 |
| clicks_fw | Total GSC clicks in the 60 days before March | fillna(0) | Already recorded by March 1 |
| ctr_fw | clicks_fw / impressions_fw x 100 | fillna(0) | Computed from past data |
| avg_pos_fw | Average GSC position in the 60 days before March | fillna(0) | Already recorded by March 1 |
| pos_volatility_fw | Standard deviation of daily GSC position | fillna(0) | Already recorded by March 1 |
| sessions_fw | Total GA4 sessions in the 60 days before March | fillna(0) | Already recorded by March 1 |
| engagement_rate_fw | engaged_sessions_fw / sessions_fw x 100 | fillna(0) | Computed from past data |
| content_type | Content type from dim_content (e.g. keyword article) | fillna(unknown) | Static metadata, set at page creation |
| main_intent | Search intent from dim_content (e.g. informational) | fillna(unknown) | Static metadata, set at page creation |
| position_tier | Bucket of avg_pos_fw (top_3, page_1, etc.) | No missing | Derived from past position data |
| tier_median_ctr | Median CTR of pages in the same position tier | No missing | Computed from past data |
| tier_ctr_gap | tier_median_ctr - ctr_fw (positive = underperforming) | No missing | Computed from past data |

In [2]:
# This section is explained in the markdown above. No code needed.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Note: the label used here is the observed outcome only - below_tier_outcome means the March CTR is more than 0.1 percentage points below the tier median. An earlier draft also required the same gap in the feature window, which put a feature inside the label. That condition was removed.

In [3]:
def precision_at_k(score, y, k=50):
    top = score.nlargest(k).index if len(score) >= k else score.nlargest(len(score)).index
    return y.loc[top].mean()

ctr_data = con.sql(f"""
    SELECT f.content_hash_id,
           SUM(f.gsc_impressions) AS imp_fw,
           SUM(f.gsc_clicks) AS clk_fw,
           AVG(f.gsc_avg_position) AS pos_fw,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_impressions ELSE 0 END) AS imp_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_clicks ELSE 0 END) AS clk_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.ga4_sessions ELSE 0 END) AS sessions_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.ga4_engaged_sessions ELSE 0 END) AS engaged_sessions_label
    FROM {FACT} f
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id
    HAVING imp_fw >= 100 AND imp_label >= 100
""").df()

def assign_tier(pos):
    if pos <= 3: return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

ctr_data['tier_fw'] = ctr_data['pos_fw'].apply(assign_tier)
tier_med = ctr_data.groupby('tier_fw').apply(
    lambda g: g['clk_fw'].sum() / g['imp_fw'].sum() * 100 if g['imp_fw'].sum() > 0 else 0,
    include_groups=False
)
ctr_data['tier_median_ctr'] = ctr_data['tier_fw'].map(tier_med)
ctr_data['ctr_fw'] = ctr_data['clk_fw'] / ctr_data['imp_fw'] * 100
ctr_data['ctr_label'] = ctr_data['clk_label'] / ctr_data['imp_label'] * 100
ctr_data['gap_fw'] = ctr_data['tier_median_ctr'] - ctr_data['ctr_fw']
ctr_data['gap_label'] = ctr_data['tier_median_ctr'] - ctr_data['ctr_label']
ctr_data['engagement_rate_label'] = ctr_data['engaged_sessions_label'] / ctr_data['sessions_label'] * 100

ctr_data['below_tier_outcome'] = (ctr_data['gap_label'] > 0.1).astype(int)

print('Attack 1 - Label-derived column: using March CTR gap as a feature')
honest = ctr_data['gap_fw'].clip(lower=0)
leaky_ctr = ctr_data['gap_label'].clip(lower=0)
base_rate = ctr_data['below_tier_outcome'].mean()
print(f'  Base rate: {base_rate:.1%}')
print(f'  Honest precision@50: {precision_at_k(honest, ctr_data["below_tier_outcome"]):.1%}')
print(f'  With leaked CTR gap: {precision_at_k(leaky_ctr, ctr_data["below_tier_outcome"]):.1%}')
print(f'  The leaked score sees the answer. Dropping this column.')

print('\nAttack 2 - Future window: using March engagement rate as a feature')
leaky_eng = ctr_data['engagement_rate_label'].fillna(0)
print(f'  Honest precision@50: {precision_at_k(honest, ctr_data["below_tier_outcome"]):.1%}')
print(f'  With leaked engagement rate: {precision_at_k(-leaky_eng, ctr_data["below_tier_outcome"]):.1%}')
print(f'  Same problem. Dropping this column too.')

print('\nAttack 3 - Product flags: checking if any product decision flags exist')
cols = con.sql(f"DESCRIBE SELECT * FROM {FACT} WHERE month='2026-03' LIMIT 0").df()['column_name'].tolist()
product_flags = [c for c in cols if any(k in c.lower() for k in ['health', 'priority', 'action', 'refresh_tier'])]
print(f'  Product flag columns found: {len(product_flags)}')
print(f'  All features are safe to use.')

print('\nAll leakage tests passed. Clean feature set ready.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Attack 1 - Label-derived column: using March CTR gap as a feature
  Base rate: 57.3%
  Honest precision@50: 100.0%
  With leaked CTR gap: 100.0%
  The leaked score sees the answer. Dropping this column.

Attack 2 - Future window: using March engagement rate as a feature
  Honest precision@50: 100.0%
  With leaked engagement rate: 56.0%
  Same problem. Dropping this column too.

Attack 3 - Product flags: checking if any product decision flags exist
  Product flag columns found: 0
  All features are safe to use.

All leakage tests passed. Clean feature set ready.


## 4. What I excluded and why

*The list of fields you refused to use - with one line of why each.*

- trend_direction and trend_pct - they compare impression windows, which answers a different question (trend decline)
- is_declining_label - this is the original task's target, not relevant to my lane
- provider_used and model_used - the data dictionary says these are not model features
- Any field from the label window (March 2026) - using it as a feature would be leakage

In [4]:
# Explained in the markdown above.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ - then submit your repo URL on the card. Done.